# Gemma 3 270M — one-minute JAX pilot on Colab Pro v5e-1

**One notebook. One TPU chip. Random initial weights.** Everything needed to run the experiment is embedded here. Setup creates an isolated Python environment in the Colab VM; it does not replace packages already loaded by the notebook kernel.

1. In Colab, select **Runtime → Change runtime type → TPU v5e-1**.
2. Choose **Runtime → Run all**.
3. Authorize Google Drive when prompted. Supply your Hugging Face token if it is not already saved in Colab Secrets as `HF_TOKEN`.

Accept the [Gemma model repository terms](https://huggingface.co/google/gemma-3-270m) first. Results and the full resumable checkpoint are saved to **My Drive / ShiningGem / colab-v5e1 / a unique run folder**. The full FP32 model plus AdamW state is approximately **3.2 GB**, so saving and verifying it takes additional time.

The **60-second interval covers training only**. Dependency installation, data preparation, model initialization, compilation, validation, and checkpoint saving have separate timings. This notebook uses your Colab runtime and Google Drive; its output does not invent a dollar price for Colab compute units.

Colab accelerator availability varies. The notebook verifies the actual device and stops if it is not exactly one v5e chip. [Colab resource documentation](https://research.google.com/colaboratory/faq.html)


## Baseline and engineering requirements

| Setting | Value |
|---|---:|
| Architecture | Native `gm.nn.Gemma3_270M` from `gemma==4.0.1` |
| Unique parameters | 268,098,176 |
| Vocabulary | Native 262,144-token vocabulary; tied input/output table |
| Initial weights | Random; seed 42 |
| Training data | 10,000 unique English FineWeb documents |
| Validation | 32 separate documents; exact-text hashes checked |
| Sequence length | 1,024 positions |
| Batch | 1 sequence per optimizer update |
| Arithmetic | BF16 forward computation; FP32 master weights, loss, and AdamW state |
| Optimizer | AdamW; LR 0.0003; 5-update warmup; gradient clipping at 1.0 |
| Timer | 60 seconds, ending at a completed optimizer update |

**COLAB-001:** The runner shall use exactly one v5e TPU device.  
**COLAB-002:** The runner shall initialize the native architecture from random weights.  
**COLAB-003:** The model shall retain its shared embedding/output table.  
**COLAB-004:** Training and validation shall share zero exact document hashes.  
**COLAB-005:** Training shall continue to the first update boundary at or after 60 seconds, with no new update starting at or after 60 seconds.  
**COLAB-006:** The runner shall complete at least 5 optimizer updates.  
**COLAB-007:** The runner shall reject nonfinite losses or gradients before applying the affected update.  
**COLAB-008:** At least one transformer parameter shall change.  
**COLAB-009:** Initial and final validation shall use identical inputs and masks.  
**COLAB-010:** The complete persistent checkpoint shall match the saved state exactly.  
**COLAB-011:** Reloaded validation loss shall match within 0.001 nats per scored token.  
**COLAB-012:** The runner shall apply zero optimizer updates outside the timed interval.

Engineering PASS requires all 12 checks. Learning PASS additionally requires lower final validation cross-entropy. Per-token loss is appropriate for this fixed-tokenizer pilot; it is not a cross-tokenizer BPB score.


## 1. Create the run folder and install the isolated environment
This cell can take several minutes on a fresh runtime. All dependency versions are embedded and pinned below. No runtime restart is needed.

In [ ]:
import os
import sys
import json
import time
import uuid
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone

NOTEBOOK_START_MONOTONIC = time.monotonic()
BASE = Path("/content/gemma_colab_pilot")
BASE.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
WORK = BASE / RUN_ID
WORK.mkdir()
if shutil.disk_usage(BASE).free < 8 * 1024**3:
    raise RuntimeError("This full-checkpoint pilot needs at least 8 GiB of free local disk.")

DEPENDENCY_LOCK = r'''
# This file was autogenerated by uv via the following command:
#    uv pip compile /tmp/gemma-colab-requirements.in --python-version 3.12 --python-platform x86_64-manylinux_2_35 -o /tmp/gemma-colab-requirements.lock
absl-py==2.5.1
    # via
    #   array-record
    #   chex
    #   clu
    #   dm-tree
    #   etils
    #   gemma
    #   grain
    #   kauldron
    #   keras
    #   ml-collections
    #   optax
    #   orbax-checkpoint
    #   tensorboard
    #   tensorflow
    #   tensorflow-datasets
    #   tensorflow-metadata
    #   xmanager
aiofile==3.12.3
    # via py-key-value-aio
aiofiles==25.1.0
    # via orbax-checkpoint
aiohappyeyeballs==2.7.1
    # via aiohttp
aiohttp==3.14.5
    # via
    #   aiohttp-retry
    #   cloud-sql-python-connector
    #   fsspec
    #   kubernetes
aiohttp-retry==2.9.1
    # via kubernetes
aiosignal==1.4.0
    # via aiohttp
alembic==1.20.0
    # via xmanager
altair==6.3.0
    # via kauldron
annotated-types==0.8.0
    # via pydantic
anyio==4.15.1
    # via
    #   google-genai
    #   httpx
    #   httpx2
    #   mcp
    #   py-key-value-aio
    #   sse-starlette
    #   starlette
    #   watchfiles
anywidget==0.11.0
    # via dialog
array-record==0.8.6
    # via
    #   grain
    #   tensorflow-datasets
asttokens==3.0.2
    # via stack-data
astunparse==1.6.3
    # via tensorflow
async-generator==1.10
    # via xmanager
attrs==26.1.0
    # via
    #   aiohttp
    #   cyclopts
    #   dm-tree
    #   jsonschema
    #   jsonschema-path
    #   referencing
    #   xmanager
authlib==1.9.1
    # via fastmcp-slim
beartype==0.23.1
    # via py-key-value-aio
cachetools==7.2.4
    # via py-key-value-aio
caio==0.12.9
    # via aiofile
certifi==2026.10.19
    # via
    #   fastmcp-slim
    #   google-cloud-aiplatform
    #   httpcore
    #   httpx
    #   kubernetes
    #   requests
cffi==2.1.1
    # via cryptography
charset-normalizer==3.5.2
    # via requests
chex==0.1.92
    # via kauldron
click==8.5.0
    # via
    #   huggingface-hub
    #   uvicorn
cloud-sql-python-connector==1.23.0
    # via xmanager
cloudpickle==3.1.2
    # via
    #   grain
    #   joblib
clu==0.0.12
    # via kauldron
comm==0.2.3
    # via ipywidgets
contourpy==1.4.0
    # via matplotlib
cryptography==50.0.2
    # via
    #   authlib
    #   cloud-sql-python-connector
    #   google-auth
    #   joserfc
    #   pyjwt
    #   secretstorage
cycler==0.12.1
    # via matplotlib
cyclopts==5.2.0
    # via fastmcp-slim
datasets==5.1.0
    # via -r /tmp/gemma-colab-requirements.in
dialog==1.1.0
    # via gemma
dill==0.4.1
    # via
    #   datasets
    #   multiprocess
distro==1.9.0
    # via google-genai
dm-tree==0.1.10
    # via tensorflow-datasets
dnspython==2.9.0
    # via
    #   cloud-sql-python-connector
    #   email-validator
docker==7.2.0
    # via xmanager
docstring-parser==0.18.0
    # via
    #   cyclopts
    #   google-cloud-aiplatform
    #   simple-parsing
durationpy==0.11
    # via kubernetes
einops==0.8.2
    # via
    #   etils
    #   gemma
    #   kauldron
email-validator==2.3.0
    # via pydantic
etils==1.14.0
    # via
    #   array-record
    #   clu
    #   dialog
    #   gemma
    #   grain
    #   kauldron
    #   orbax-checkpoint
    #   tensorflow-datasets
    #   xmanager
exceptiongroup==1.3.1
    # via fastmcp-slim
executing==2.3.0
    # via stack-data
fastmcp==4.1.3
    # via gemma
fastmcp-slim==4.1.3
    # via fastmcp
filelock==4.2.4
    # via
    #   datasets
    #   huggingface-hub
flatbuffers==25.12.19
    # via tensorflow
flax==0.12.10
    # via
    #   -r /tmp/gemma-colab-requirements.in
    #   clu
    #   gemma
    #   kauldron
fonttools==4.67.0
    # via matplotlib
frozenlist==1.8.0
    # via
    #   aiohttp
    #   aiosignal
fsspec==2026.7.0
    # via
    #   datasets
    #   etils
    #   huggingface-hub
gast==0.7.0
    # via tensorflow
gemma==4.0.1
    # via -r /tmp/gemma-colab-requirements.in
gitdb==4.0.12
    # via gitpython
gitpython==3.2.1
    # via xmanager
google-api-core==2.42.0
    # via
    #   google-api-python-client
    #   google-cloud-aiplatform
    #   google-cloud-bigquery
    #   google-cloud-core
    #   google-cloud-resource-manager
    #   google-cloud-sql
    #   google-cloud-storage
    #   xmanager
google-api-python-client==2.201.0
    # via xmanager
google-auth==2.62.0
    # via
    #   cloud-sql-python-connector
    #   google-api-core
    #   google-api-python-client
    #   google-auth-httplib2
    #   google-cloud-aiplatform
    #   google-cloud-bigquery
    #   google-cloud-core
    #   google-cloud-resource-manager
    #   google-cloud-sql
    #   google-cloud-storage
    #   google-genai
    #   xmanager
google-auth-httplib2==0.4.4
    # via google-api-python-client
google-cloud-aiplatform==2.6.0
    # via xmanager
google-cloud-bigquery==3.46.2
    # via google-cloud-aiplatform
google-cloud-core==2.8.0
    # via
    #   google-cloud-bigquery
    #   google-cloud-storage
google-cloud-resource-manager==1.19.0
    # via google-cloud-aiplatform
google-cloud-sql==0.1.1
    # via cloud-sql-python-connector
google-cloud-storage==3.17.1
    # via
    #   google-cloud-aiplatform
    #   xmanager
google-crc32c==1.9.0
    # via
    #   google-cloud-storage
    #   google-resumable-media
google-genai==2.32.0
    # via google-cloud-aiplatform
google-pasta==0.2.0
    # via tensorflow
google-resumable-media==2.11.0
    # via
    #   google-cloud-bigquery
    #   google-cloud-storage
googleapis-common-protos==1.75.5
    # via
    #   google-api-core
    #   grpc-google-iam-v1
    #   grpcio-status
    #   tensorflow-metadata
    #   xmanager
grain==0.2.18
    # via
    #   gemma
    #   kauldron
graphviz==0.21
    # via kauldron
griffelib==2.4.0
    # via fastmcp-slim
grpc-google-iam-v1==0.14.5
    # via google-cloud-resource-manager
grpcio==1.84.0
    # via
    #   google-api-core
    #   google-cloud-resource-manager
    #   google-cloud-sql
    #   googleapis-common-protos
    #   grpc-google-iam-v1
    #   grpcio-status
    #   tensorboard
    #   tensorflow
    #   xmanager
grpcio-status==1.84.0
    # via google-api-core
h11==0.16.0
    # via
    #   httpcore
    #   httpcore2
    #   uvicorn
h5py==3.14.0
    # via
    #   keras
    #   tensorflow
hf-xet==1.7.0
    # via huggingface-hub
httpcore==1.0.9
    # via httpx
httpcore2==2.16.0
    # via httpx2
httplib2==0.32.0
    # via
    #   google-api-python-client
    #   google-auth-httplib2
httpx==0.28.1
    # via google-genai
httpx2==2.16.0
    # via
    #   fastmcp-slim
    #   huggingface-hub
    #   mcp
huggingface-hub==2.2.0
    # via
    #   -r /tmp/gemma-colab-requirements.in
    #   datasets
    #   tokenizers
humanize==4.16.0
    # via
    #   orbax-checkpoint
    #   xmanager
idna==3.20
    # via
    #   anyio
    #   email-validator
    #   httpx
    #   httpx2
    #   requests
    #   yarl
imageio==2.38.1
    # via scikit-image
immutabledict==4.3.1
    # via
    #   kauldron
    #   tensorflow-datasets
    #   xmanager
ipython==9.17.1
    # via
    #   ipywidgets
    #   mediapy
ipython-pygments-lexers==1.1.1
    # via ipython
ipywidgets==8.1.9
    # via anywidget
jaraco-classes==3.4.0
    # via keyring
jaraco-context==6.1.2
    # via keyring
jaraco-functools==4.6.0
    # via keyring
jax==0.12.0
    # via
    #   -r /tmp/gemma-colab-requirements.in
    #   chex
    #   clu
    #   flax
    #   gemma
    #   kauldron
    #   optax
    #   orbax-checkpoint
jaxlib==0.12.0
    # via
    #   -r /tmp/gemma-colab-requirements.in
    #   chex
    #   clu
    #   jax
    #   optax
jaxtyping==0.3.11
    # via
    #   gemma
    #   kauldron
jedi==0.20.1
    # via ipython
jeepney==0.9.0
    # via
    #   keyring
    #   secretstorage
jinja2==3.1.6
    # via altair
joblib==1.6.0
    # via scikit-learn
joserfc==1.7.7
    # via
    #   authlib
    #   fastmcp-slim
jsonref==1.1.0
    # via fastmcp-slim
jsonschema==4.26.0
    # via
    #   altair
    #   mcp
jsonschema-path==0.5.0
    # via fastmcp-slim
jsonschema-specifications==2025.9.1
    # via jsonschema
jupyterlab-widgets==3.0.17
    # via ipywidgets
kauldron==1.4.2
    # via
    #   -r /tmp/gemma-colab-requirements.in
    #   gemma
keras==3.15.1
    # via tensorflow
keyring==25.7.0
    # via py-key-value-aio
kiwisolver==1.5.1
    # via matplotlib
kubernetes==37.0.1
    # via xmanager
lark==1.3.1
    # via
    #   dialog
    #   kauldron
lazy-loader==0.6
    # via scikit-image
libclang==18.1.1
    # via tensorflow
libtpu==0.0.50
    # via jax
mako==1.4.3
    # via alembic
markdown==3.11.2
    # via tensorboard
markdown-it-py==4.2.0
    # via rich
markupsafe==3.0.4
    # via
    #   jinja2
    #   mako
    #   werkzeug
matplotlib==3.11.2
    # via mediapy
matplotlib-inline==0.2.2
    # via ipython
mcp==2.3.0
    # via
    #   dialog
    #   fastmcp-slim
    #   gemma
mcp-types==2.3.0
    # via
    #   fastmcp-slim
    #   mcp
mdurl==0.1.2
    # via markdown-it-py
mediapy==1.2.7
    # via kauldron
ml-collections==1.1.0
    # via
    #   clu
    #   kauldron
ml-dtypes==0.6.0
    # via
    #   jax
    #   jaxlib
    #   keras
    #   tensorflow
    #   tensorstore
more-itertools==11.2.1
    # via
    #   jaraco-classes
    #   jaraco-functools
msgpack==1.2.3
    # via
    #   flax
    #   orbax-checkpoint
multidict==7.1.0
    # via
    #   aiohttp
    #   yarl
multiprocess==0.70.19
    # via datasets
namex==0.1.0
    # via keras
narwhals==2.27.1
    # via
    #   altair
    #   scikit-learn
networkx==3.7
    # via scikit-image
numpy==2.5.4
    # via
    #   -r /tmp/gemma-colab-requirements.in
    #   chex
    #   clu
    #   contourpy
    #   datasets
    #   dialog
    #   dm-tree
    #   etils
    #   flax
    #   gemma
    #   grain
    #   h5py
    #   imageio
    #   jax
    #   jaxlib
    #   kauldron
    #   keras
    #   matplotlib
    #   mediapy
    #   ml-dtypes
    #   opencv-python
    #   optax
    #   orbax-checkpoint
    #   pandas
    #   scikit-image
    #   scikit-learn
    #   scipy
    #   tensorboard
    #   tensorflow
    #   tensorflow-datasets
    #   tensorstore
    #   tifffile
    #   treescope
oauthlib==4.0.0
    # via requests-oauthlib
openapi-pydantic==0.6.0
    # via fastmcp-slim
opencv-python==5.0.0.93
    # via kauldron
opentelemetry-api==1.45.1
    # via
    #   fastmcp-slim
    #   google-api-core
    #   mcp
opt-einsum==3.4.0
    # via
    #   jax
    #   tensorflow
optax==0.2.8
    # via
    #   -r /tmp/gemma-colab-requirements.in
    #   flax
    #   kauldron
optree==0.20.0
    # via keras
orbax-checkpoint==0.12.8
    # via
    #   flax
    #   gemma
    #   kauldron
packaging==26.3
    # via
    #   altair
    #   clu
    #   datasets
    #   fastmcp-slim
    #   google-cloud-aiplatform
    #   google-cloud-bigquery
    #   huggingface-hub
    #   keras
    #   lazy-loader
    #   matplotlib
    #   scikit-image
    #   tensorboard
    #   tensorflow
    #   wheel
pandas==3.0.6
    # via
    #   datasets
    #   kauldron
parso==0.8.7
    # via jedi
pathable==0.6.0
    # via jsonschema-path
pexpect==4.9.0
    # via ipython
pillow==13.0.0
    # via
    #   dialog
    #   imageio
    #   matplotlib
    #   mediapy
    #   scikit-image
    #   tensorboard
platformdirs==4.13.5
    # via fastmcp-slim
portpicker==1.6.0
    # via grain
prometheus-client==0.26.0
    # via orbax-checkpoint
promise==2.3
    # via tensorflow-datasets
prompt-toolkit==3.0.53
    # via ipython
propcache==0.5.4
    # via
    #   aiohttp
    #   yarl
proto-plus==1.29.0
    # via
    #   google-api-core
    #   google-cloud-aiplatform
    #   google-cloud-resource-manager
    #   google-cloud-sql
protobuf==6.33.6
    # via
    #   google-api-core
    #   google-cloud-aiplatform
    #   google-cloud-resource-manager
    #   google-cloud-sql
    #   googleapis-common-protos
    #   grain
    #   grpc-google-iam-v1
    #   grpcio-status
    #   orbax-checkpoint
    #   proto-plus
    #   tensorboard
    #   tensorflow
    #   tensorflow-datasets
    #   tensorflow-metadata
    #   xmanager
psutil==7.2.2
    # via
    #   ipython
    #   orbax-checkpoint
    #   portpicker
    #   tensorflow-datasets
    #   xmanager
psygnal==0.16.1
    # via anywidget
ptyprocess==0.7.0
    # via pexpect
pure-eval==0.2.4
    # via stack-data
py-key-value-aio==0.4.6
    # via fastmcp-slim
pyarrow==26.0.0
    # via
    #   datasets
    #   tensorflow-datasets
pyasn1==0.6.4
    # via pyasn1-modules
pyasn1-modules==0.4.2
    # via google-auth
pycparser==3.11
    # via cffi
pydantic==2.14.1
    # via
    #   fastmcp-slim
    #   google-cloud-aiplatform
    #   google-genai
    #   kubernetes
    #   mcp
    #   mcp-types
    #   openapi-pydantic
    #   pydantic-settings
pydantic-core==2.50.1
    # via pydantic
pydantic-settings==2.16.1
    # via fastmcp-slim
pydub==0.25.1
    # via dialog
pygments==2.21.0
    # via
    #   ipython
    #   ipython-pygments-lexers
    #   rich
    #   rich-rst
pyjwt==2.15.1
    # via mcp
pyparsing==3.3.3
    # via
    #   httplib2
    #   matplotlib
pyperclip==1.11.0
    # via fastmcp-slim
python-dateutil==2.9.0.post0
    # via
    #   google-cloud-bigquery
    #   kubernetes
    #   matplotlib
    #   pandas
python-dotenv==1.2.4
    # via
    #   fastmcp-slim
    #   pydantic-settings
    #   xmanager
python-multipart==0.0.32
    # via
    #   fastmcp-slim
    #   mcp
pyyaml==6.0.3
    # via
    #   datasets
    #   fastmcp-slim
    #   flax
    #   huggingface-hub
    #   jsonschema-path
    #   kubernetes
    #   ml-collections
    #   orbax-checkpoint
    #   xmanager
referencing==0.37.0
    # via
    #   jsonschema
    #   jsonschema-path
    #   jsonschema-specifications
requests==2.34.2
    # via
    #   cloud-sql-python-connector
    #   dialog
    #   docker
    #   google-api-core
    #   google-auth
    #   google-cloud-bigquery
    #   google-cloud-storage
    #   google-genai
    #   jax
    #   kubernetes
    #   requests-oauthlib
    #   tensorflow
    #   tensorflow-datasets
    #   xmanager
requests-oauthlib==2.0.0
    # via kubernetes
rich==15.0.0
    # via
    #   cyclopts
    #   fastmcp-slim
    #   flax
    #   keras
    #   rich-rst
rich-rst==2.2.0
    # via cyclopts
rpds-py==2026.9.1
    # via
    #   jsonschema
    #   referencing
scikit-image==0.26.0
    # via kauldron
scikit-learn==1.9.1
    # via kauldron
scipy==1.18.1
    # via
    #   jax
    #   jaxlib
    #   scikit-image
    #   scikit-learn
secretstorage==3.5.0
    # via keyring
sentencepiece==0.2.2
    # via gemma
setuptools==84.2.1
    # via
    #   tensorboard
    #   tensorflow
simple-parsing==0.1.9
    # via tensorflow-datasets
simplejson==4.2.0
    # via orbax-checkpoint
six==1.17.0
    # via
    #   astunparse
    #   google-pasta
    #   promise
    #   python-dateutil
    #   tensorflow
smmap==5.0.3
    # via gitdb
sniffio==1.3.1
    # via google-genai
sqlalchemy==2.1.5
    # via
    #   alembic
    #   xmanager
sqlparse==0.6.0
    # via xmanager
sse-starlette==3.5.0
    # via mcp
stack-data==0.6.3
    # via ipython
starlette==1.9.0
    # via
    #   fastmcp-slim
    #   mcp
    #   sse-starlette
tabulate==0.10.0
    # via
    #   kauldron
    #   xmanager
tenacity==9.1.4
    # via google-genai
tensorboard==2.21.0
    # via kauldron
tensorboard-data-server==0.7.2
    # via tensorboard
tensorflow==2.21.0
    # via kauldron
tensorflow-datasets==4.9.10
    # via kauldron
tensorflow-metadata==1.21.0
    # via tensorflow-datasets
tensorstore==0.1.85
    # via
    #   flax
    #   orbax-checkpoint
termcolor==3.3.0
    # via
    #   tensorflow
    #   tensorflow-datasets
    #   xmanager
threadpoolctl==3.7.0
    # via scikit-learn
tifffile==2026.10.15
    # via scikit-image
tokenizers==0.23.3
    # via -r /tmp/gemma-colab-requirements.in
toml==0.10.2
    # via tensorflow-datasets
toolz==1.2.0
    # via chex
tqdm==4.70.1
    # via
    #   datasets
    #   etils
    #   huggingface-hub
    #   kauldron
    #   tensorflow-datasets
traitlets==5.16.1
    # via
    #   ipython
    #   ipywidgets
    #   matplotlib-inline
treescope==0.1.10
    # via
    #   flax
    #   gemma
truststore==0.10.5
    # via
    #   httpcore2
    #   httpx2
typeguard==4.6.0
    # via kauldron
typing-extensions==4.16.0
    # via
    #   aiohttp
    #   aiosignal
    #   alembic
    #   altair
    #   anyio
    #   anywidget
    #   chex
    #   clu
    #   etils
    #   exceptiongroup
    #   fastmcp-slim
    #   flax
    #   google-cloud-aiplatform
    #   google-genai
    #   grpcio
    #   httpx2
    #   huggingface-hub
    #   kauldron
    #   kubernetes
    #   mcp
    #   mcp-types
    #   opentelemetry-api
    #   optree
    #   orbax-checkpoint
    #   py-key-value-aio
    #   pydantic
    #   pydantic-core
    #   pydantic-settings
    #   referencing
    #   simple-parsing
    #   sqlalchemy
    #   starlette
    #   tensorflow
    #   typeguard
    #   typing-inspection
typing-inspection==0.4.4
    # via
    #   mcp
    #   pydantic
    #   pydantic-settings
uncalled-for==0.4.1
    # via fastmcp-slim
uritemplate==4.2.0
    # via google-api-python-client
urllib3==2.8.0
    # via
    #   docker
    #   kubernetes
    #   requests
uvicorn==0.54.0
    # via
    #   fastmcp-slim
    #   mcp
uvloop==0.23.0
    # via orbax-checkpoint
wadler-lindig==0.1.7
    # via jaxtyping
watchfiles==1.3.0
    # via fastmcp-slim
wcwidth==0.9.3
    # via prompt-toolkit
websocket-client==1.9.2
    # via kubernetes
websockets==16.1.1
    # via
    #   fastmcp-slim
    #   google-genai
werkzeug==3.1.9
    # via tensorboard
wheel==0.48.0
    # via astunparse
widgetsnbextension==4.0.16
    # via ipywidgets
wrapt==2.5.1
    # via
    #   clu
    #   dm-tree
    #   tensorflow
    #   tensorflow-datasets
xmanager==0.8.4
    # via kauldron
xxhash==4.0.2
    # via datasets
yarl==1.25.1
    # via aiohttp
zipp==4.1.1
    # via etils

'''

LOCK_PATH = WORK / "requirements.lock"
LOCK_PATH.write_text(DEPENDENCY_LOCK)
TOOLING = BASE / ".bootstrap"
if not (TOOLING / "uv").is_dir():
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--target", str(TOOLING), "uv==0.11.15"], check=True)
uv_env = dict(os.environ, PYTHONPATH=str(TOOLING))
uv = [sys.executable, "-m", "uv"]
PYTHON = BASE / ".venv" / "bin" / "python"
if not PYTHON.exists():
    subprocess.run([*uv, "venv", "--python", "3.12", "--managed-python", str(PYTHON.parent.parent)], env=uv_env, check=True)
print("Installing the pinned Gemma/JAX environment...", flush=True)
subprocess.run([*uv, "pip", "install", "--quiet", "--python", str(PYTHON), "-r", str(LOCK_PATH)], env=uv_env, check=True)
SETUP_SECONDS = time.monotonic() - NOTEBOOK_START_MONOTONIC
print(f"Environment ready. Installation/setup: {SETUP_SECONDS:.1f} seconds.")


## 2. Connect Google Drive and authenticate the tokenizer download
The Hugging Face token is passed only to preparation. It is not printed or saved in the checkpoint, notebook outputs, or result files.

In [ ]:
from google.colab import drive, userdata
from getpass import getpass

drive.mount("/content/drive")
OUTPUT = Path("/content/drive/MyDrive/ShiningGem/colab-v5e1") / RUN_ID
OUTPUT.mkdir(parents=True, exist_ok=False)
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass("Hugging Face token with access to google/gemma-3-270m: ")
if not HF_TOKEN.strip():
    raise ValueError("A Hugging Face token with accepted Gemma repository access is required.")
print("Persistent results folder:", OUTPUT)


## 3. Define the complete pilot
This embedded program is written automatically into the temporary Colab workspace. You do not need to upload or edit another file. The native model comes from the pinned Gemma library; no pretrained model weights are downloaded.

In [ ]:
PROGRAM_SOURCE = r'''
"""Self-contained one-chip Colab Gemma pilot, embedded in its notebook."""
from __future__ import annotations
import argparse
import dataclasses
import gc
import hashlib
import importlib.metadata
import json
import math
import os
from pathlib import Path
import shutil
import sys
import time
import traceback
import zipfile
import numpy as np

def read_json(path):
    return json.loads(Path(path).read_text())

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\n")
    temporary.replace(path)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def chunk_document(ids, sequence_length, bos_id, eos_id, pad_id=0):
    """Every target appears once; consecutive chunks overlap by one input token."""
    if pad_id in ids:
        raise ValueError("Document encodes a padding ID; cannot silently mask real text.")
    sequence = [bos_id, *ids, eos_id]
    for start in range(0, len(sequence) - 1, sequence_length):
        window = sequence[start:start + sequence_length + 1]
        count = len(window) - 1
        tokens = np.full(sequence_length, pad_id, np.int32)
        targets = np.full(sequence_length, pad_id, np.int32)
        mask = np.zeros(sequence_length, np.bool_)
        tokens[:count] = window[:-1]
        targets[:count] = window[1:]
        mask[:count] = True
        yield tokens, targets, mask


def prepare(config, destination):
    from datasets import load_dataset
    from huggingface_hub import HfApi, snapshot_download
    from tokenizers import Tokenizer

    started = time.monotonic()
    destination = Path(destination)
    if destination.exists():
        raise FileExistsError(f"Refusing to replace prepared data: {destination}")
    destination.mkdir(parents=True)
    api = HfApi()
    model_sha = api.model_info(config["model_id"], revision=config["model_revision"]).sha
    data_sha = api.dataset_info(config["dataset_id"], revision=config["dataset_revision"]).sha
    tokenizer_dir = destination / "tokenizer"
    snapshot_download(config["model_id"], revision=model_sha, local_dir=tokenizer_dir,
                      allow_patterns=["config.json", "tokenizer.json", "tokenizer_config.json",
                                      "special_tokens_map.json", "tokenizer.model"])
    tokenizer = Tokenizer.from_file(str(tokenizer_dir / "tokenizer.json"))
    model_config = read_json(tokenizer_dir / "config.json")
    text_config = model_config.get("text_config", model_config)
    expected = {"hidden_size": 640, "intermediate_size": 2048, "num_hidden_layers": 18,
                "num_attention_heads": 4, "num_key_value_heads": 1, "head_dim": 256,
                "vocab_size": 262144}
    for key, value in expected.items():
        if text_config.get(key) != value:
            raise ValueError(f"Unexpected native model setting {key}: {text_config.get(key)}")
    bos = text_config["bos_token_id"]
    eos = text_config["eos_token_id"]
    if isinstance(eos, list):
        eos = eos[0]
    pad = text_config.get("pad_token_id", 0)
    if pad != 0:
        raise ValueError("Gemma library expects padding ID 0.")
    stream = load_dataset(config["dataset_id"], config["dataset_config"], split="train",
                          revision=data_sha, streaming=True)
    selected = {"train": [], "validation": []}
    seen = set()
    desired = config["train_documents"] + config["validation_documents"]
    scanned = 0
    for document in stream:
        scanned += 1
        text = document["text"]
        if not text.strip():
            continue
        digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
        if digest in seen:
            continue
        seen.add(digest)
        split = "validation" if len(selected["validation"]) < config["validation_documents"] else "train"
        selected[split].append({"id": str(document.get("id", digest)), "sha256": digest, "text": text})
        if len(seen) == desired:
            break
    if len(seen) != desired:
        raise RuntimeError(f"Only found {len(seen)} unique nonempty documents; need {desired}.")
    manifest = {"model_id": config["model_id"], "model_revision": model_sha,
                "dataset_id": config["dataset_id"], "dataset_config": config["dataset_config"],
                "dataset_revision": data_sha, "documents_scanned": scanned,
                "sequence_length": config["sequence_length"], "special_ids": {"bos": bos, "eos": eos, "pad": pad},
                "native_config": expected, "splits": {}, "files": {}}
    for split, documents in selected.items():
        rows, metadata, roundtrips = [], [], 0
        with (destination / f"{split}.jsonl").open("w", encoding="utf-8") as output:
            for document_index, document in enumerate(documents):
                output.write(json.dumps(document, ensure_ascii=False) + "\n")
                ids = tokenizer.encode(document["text"], add_special_tokens=False).ids
                roundtrips += tokenizer.decode(ids, skip_special_tokens=False) == document["text"]
                if not ids or min(ids) < 0 or max(ids) >= expected["vocab_size"]:
                    raise ValueError(f"Invalid token IDs for {split} document {document_index}.")
                rows.extend((*chunk, document_index) for chunk in chunk_document(
                    ids, config["sequence_length"], bos, eos, pad))
                metadata.append({key: document[key] for key in ("id", "sha256")})
        tokens, targets, masks, doc_indices = zip(*rows)
        arrays = {"tokens": np.stack(tokens), "targets": np.stack(targets),
                  "mask": np.stack(masks), "document_index": np.array(doc_indices, np.int32)}
        np.savez_compressed(destination / f"{split}.npz", **arrays)
        manifest["splits"][split] = {"documents": metadata, "blocks": len(rows),
            "scored_tokens": int(arrays["mask"].sum()), "exact_roundtrip_documents": roundtrips,
            "original_utf8_bytes": sum(len(doc["text"].encode("utf-8")) for doc in documents)}
        print(f"{split}: {len(documents)} documents, {len(rows)} blocks", flush=True)
    if manifest["splits"]["train"]["blocks"] < config["expected_devices"]:
        raise ValueError("Need at least one distinct training block per chip.")
    for path in sorted(destination.rglob("*")):
        if path.is_file() and ".cache" not in path.parts:
            manifest["files"][str(path.relative_to(destination))] = sha256_file(path)
    manifest["preparation_seconds"] = time.monotonic() - started
    write_json(destination / "manifest.json", manifest)
    write_json(destination / "resolved_config.json", {**config, "model_revision": model_sha, "dataset_revision": data_sha})
    return manifest


MODEL_HELPERS = '@struct.dataclass\nclass PilotState:\n    step: jax.Array\n    params: object\n    opt_state: object\n    rng: jax.Array\n    data_position: jax.Array\n\n\ndef make_model(cpu_fixture=False):\n    # Kauldron\'s dtype registry assumes Linux\'s extended-precision NumPy alias.\n    # This compatibility alias is used only by the explicitly marked Mac CPU fixture.\n    if cpu_fixture and not hasattr(np, "float128"):\n        np.float128 = np.longdouble\n    from gemma import gm\n    model = gm.nn.Gemma3_270M(dtype=jnp.float32)\n    if cpu_fixture:\n        config = dataclasses.replace(model.config, num_embed=128, embed_dim=32,\n            hidden_dim=64, num_heads=2, head_dim=16, num_kv_heads=1,\n            attention_types=model.config.attention_types[:2], sliding_window_size=8)\n        model = model.clone(config=config)\n    return model\n\n\ndef make_optimizer(config, params):\n    schedule = optax.join_schedules([\n        optax.linear_schedule(0.0, config["learning_rate"], config["warmup_updates"]),\n        optax.constant_schedule(config["learning_rate"]),\n    ], [config["warmup_updates"]])\n    mask = jax.tree.map(lambda value: value.ndim > 1, params)\n    optimizer = optax.chain(optax.clip_by_global_norm(config["gradient_clip"]),\n        optax.adamw(schedule, b1=config["adam_b1"], b2=config["adam_b2"],\n                    eps=config["adam_eps"], weight_decay=config["weight_decay"], mask=mask))\n    return optimizer, schedule\n\n\ndef loss_parts(model, params, batch):\n    # Master weights remain FP32. Only the forward computation sees BF16 copies.\n    compute_params = jax.tree.map(lambda value: value.astype(jnp.bfloat16), params)\n    logits = model.apply({"params": compute_params}, tokens=batch["tokens"]).logits\n    logits = logits.astype(jnp.float32)\n    target_logits = jnp.take_along_axis(logits, batch["targets"][..., None], axis=-1)[..., 0]\n    losses = jax.scipy.special.logsumexp(logits, axis=-1) - target_logits\n    weights = batch["mask"].astype(jnp.float32)\n    return jnp.sum(jnp.where(weights > 0, losses, 0.0)), jnp.sum(weights)\n'

def load_model_helpers(cpu_test=False):
    accelerator_imports(cpu_test)
    exec(MODEL_HELPERS, globals())



def accelerator_imports(cpu_test=False):
    global jax, jnp, optax, struct, serialization, traverse_util
    os.environ["JAX_PLATFORMS"] = "cpu" if cpu_test else "tpu"
    import jax
    import jax.numpy as jnp
    import optax
    from flax import struct, serialization, traverse_util
    return jax


def preflight(cpu_test=False):
    accelerator_imports(cpu_test)
    devices = jax.devices()
    if len(devices) != 1:
        raise RuntimeError(f"This notebook requires exactly ONE v5e chip; found {len(devices)} devices: {devices}. Select the v5e-1 TPU runtime in Colab.")
    device = devices[0]
    if not cpu_test:
        kind = device.device_kind.lower().replace(" ", "")
        if device.platform != "tpu" or not any(label in kind for label in ("v5lite", "v5e")):
            raise RuntimeError(f"Expected a v5e-1 TPU, found {device}. Change the Colab runtime hardware accelerator.")
    return {"platform": device.platform, "device_kind": device.device_kind,
            "device_count": len(devices), "cpu_test": cpu_test,
            "python": sys.version, "jax": jax.__version__,
            "packages": {d.metadata["Name"]: d.version for d in importlib.metadata.distributions()}}


def numpy_leaf(value):
    if isinstance(value, jax.Array):
        return np.asarray(jax.device_get(value))
    return np.asarray(value)


def archive_tree(state_dict):
    leaves = []
    def encode(node):
        if isinstance(node, dict):
            return {key: encode(value) for key, value in node.items()}
        name = f"arrays/{len(leaves):05d}.npy"
        leaves.append((name, node))
        return {"__array__": name}
    return encode(state_dict), leaves


def save_archive(path, state, metadata):
    """Write one array at a time; never materialize a full host checkpoint copy."""
    tree, leaves = archive_tree(serialization.to_state_dict(state))
    temporary = Path(str(path) + ".partial")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
        for name, leaf in leaves:
            array = numpy_leaf(leaf)
            with archive.open(name, "w", force_zip64=True) as stream:
                np.lib.format.write_array(stream, array, allow_pickle=False)
            del array
        archive.writestr("checkpoint.json", json.dumps({"tree": tree, "metadata": metadata}, allow_nan=False))
    temporary.replace(path)
    return len(leaves)


def verify_archive(path, state):
    """Compare the persistent checkpoint with current state, one array at a time."""
    tree, leaves = archive_tree(serialization.to_state_dict(state))
    with zipfile.ZipFile(path) as archive:
        description = json.loads(archive.read("checkpoint.json"))
        if tree != description["tree"]:
            raise ValueError("Checkpoint tree differs from the original state.")
        for name, leaf in leaves:
            with archive.open(name) as stream:
                stored = np.lib.format.read_array(stream, allow_pickle=False)
            actual = numpy_leaf(leaf)
            if stored.dtype != actual.dtype or not np.array_equal(stored, actual):
                raise ValueError(f"Checkpoint state mismatch: {name}")
            del stored, actual
    return len(leaves)


def restore_archive(path, abstract_state):
    """Restore arrays directly to the TPU before reconstructing the state class."""
    with zipfile.ZipFile(path) as archive:
        description = json.loads(archive.read("checkpoint.json"))
        def decode(node):
            if set(node) == {"__array__"}:
                with archive.open(node["__array__"]) as stream:
                    array = np.lib.format.read_array(stream, allow_pickle=False)
                value = jax.device_put(array)
                jax.block_until_ready(value)
                return value
            return {key: decode(value) for key, value in node.items()}
        restored_dict = decode(description["tree"])
    return serialization.from_state_dict(abstract_state, restored_dict)


def create_fixture(directory, config):
    directory.mkdir(parents=True, exist_ok=False)
    splits = {}
    for split, count in (("train",64),("validation",32)):
        rows = []
        docs = []
        for index in range(count):
            ids = [3 + index % 7] * 23
            rows.extend((*row,index) for row in chunk_document(ids,16,2,1))
            docs.append({"id":f"{split}-{index}","sha256":hashlib.sha256(f"{split}-{index}".encode()).hexdigest()})
        x,y,m,indices = zip(*rows)
        np.savez_compressed(directory/f"{split}.npz", tokens=np.stack(x),targets=np.stack(y),
                            mask=np.stack(m),document_index=np.array(indices,np.int32))
        splits[split] = {"documents":docs,"blocks":len(rows)}
    manifest = {"splits":splits,"cpu_test":True,"files":{}}
    for name in ("train.npz","validation.npz"):
        manifest["files"][name] = sha256_file(directory/name)
    write_json(directory/"manifest.json",manifest)
    write_json(directory/"resolved_config.json",{**config,"sequence_length":16,"training_seconds":0.5,"minimum_updates":5})


def train_pilot(work, output, cpu_test=False):
    environment = preflight(cpu_test)
    load_model_helpers(cpu_test)
    work, output = Path(work), Path(output)
    output.mkdir(parents=True, exist_ok=True)
    data_dir = work/"prepared"
    if not cpu_test and shutil.disk_usage(work).free < 4*1024**3:
        raise RuntimeError("At least 4 GiB of free local disk is required for the resumable checkpoint.")
    config = read_json(data_dir/"resolved_config.json")
    manifest = read_json(data_dir/"manifest.json")
    manifest_hash = sha256_file(data_dir/"manifest.json")
    write_json(output/"environment.json",environment)
    write_json(output/"config.json",config)
    for filename,digest in manifest["files"].items():
        if sha256_file(data_dir/filename) != digest:
            raise ValueError(f"Prepared input changed: {filename}")
    train_hashes = {d["sha256"] for d in manifest["splits"]["train"]["documents"]}
    eval_hashes = {d["sha256"] for d in manifest["splits"]["validation"]["documents"]}
    if train_hashes & eval_hashes:
        raise ValueError("Training and validation document hashes overlap.")
    with np.load(data_dir/"train.npz",allow_pickle=False) as archive:
        training = {key:archive[key] for key in ("tokens","targets","mask")}
    with np.load(data_dir/"validation.npz",allow_pickle=False) as archive:
        validation = {key:archive[key] for key in ("tokens","targets","mask")}
    order = np.random.default_rng(config["seed"]).permutation(len(training["tokens"]))
    model = make_model(cpu_test)
    print("Initializing random weights...",flush=True)
    init_started = time.monotonic()
    params = jax.jit(lambda key:model.init(key,tokens=jnp.ones((1,1),jnp.int32))["params"])(jax.random.PRNGKey(config["seed"]))
    jax.block_until_ready(params)
    flat = traverse_util.flatten_dict(params)
    parameter_count = sum(leaf.size for leaf in flat.values())
    if not cpu_test and parameter_count != 268_098_176:
        raise ValueError(f"Unexpected native parameter count: {parameter_count}")
    if not all(leaf.dtype == jnp.float32 for leaf in flat.values()):
        raise TypeError("Master parameters must be FP32.")
    embedding_shape = (model.config.num_embed,model.config.embed_dim)
    tied = (flat[("embedder","input_embedding")].shape == embedding_shape
            and sum(leaf.shape == embedding_shape for leaf in flat.values()) == 1)
    probe_path = next(path for path,leaf in sorted(flat.items()) if path[0].startswith("layer_") and leaf.ndim>1)
    def probe(params):
        leaf = traverse_util.flatten_dict(params)[probe_path]
        return np.array(jax.device_get(leaf.reshape(-1)[:64]))
    initial_probe = probe(params)
    optimizer,schedule = make_optimizer(config,params)
    state = PilotState(jnp.array(0,jnp.int32),params,optimizer.init(params),
                       jax.random.PRNGKey(config["seed"]),jnp.array(0,jnp.int32))
    del flat,params
    jax.block_until_ready(state)
    initialization_seconds = time.monotonic()-init_started
    def training_step(state,batch):
        def objective(params):
            total,count = loss_parts(model,params,batch)
            return total/jnp.maximum(count,1.0),count
        (loss,count),grads = jax.value_and_grad(objective,has_aux=True)(state.params)
        grad_norm = optax.tree.norm(grads)
        finite = jnp.isfinite(loss)&jnp.isfinite(grad_norm)&(count>0)
        def apply(_):
            updates,opt_state = optimizer.update(grads,state.opt_state,state.params)
            return state.replace(step=state.step+1,params=optax.apply_updates(state.params,updates),
                                 opt_state=opt_state,data_position=state.data_position+1)
        updated = jax.lax.cond(finite,apply,lambda _:state,operand=None)
        return updated,{"loss":loss,"tokens":count,"gradient_norm":grad_norm,
                        "finite":finite,"learning_rate":schedule(state.step)}
    training_jit = jax.jit(training_step,donate_argnums=(0,))
    evaluation_jit = jax.jit(lambda params,batch:loss_parts(model,params,batch))
    def batch_at(arrays,index):
        return {key:jax.device_put(value[index:index+1]) for key,value in arrays.items()}
    example = batch_at(training,0)
    print(f"Parameters: {parameter_count:,}; compiling train/evaluation (no weight updates)...",flush=True)
    compile_started = time.monotonic()
    training_executable = training_jit.lower(state,example).compile()
    evaluation_executable = evaluation_jit.lower(state.params,example).compile()
    compilation_seconds = time.monotonic()-compile_started
    def evaluate(params):
        total,count = 0.0,0
        for index in range(len(validation["tokens"])):
            nll,tokens = jax.block_until_ready(evaluation_executable(params,batch_at(validation,index)))
            total += float(nll);count += int(tokens)
        if count==0:
            raise ValueError("Validation has zero scored targets.")
        return {"loss":total/count,"nll_sum":total,"scored_tokens":count,"manifest_sha256":manifest_hash}
    print("Evaluating the initial model...",flush=True)
    phase = time.monotonic();before = evaluate(state.params);initial_eval_seconds = time.monotonic()-phase
    jax.block_until_ready(state)
    print(f"Starting {config['training_seconds']:.1f}-second training interval...",flush=True)
    started = time.monotonic()
    updates=0;tokens_processed=0;last_start=0.0
    log_path=work/"training.jsonl"
    with log_path.open("w") as log:
        while True:
            decision_elapsed=time.monotonic()-started
            if decision_elapsed>=config["training_seconds"]:
                break
            last_start=decision_elapsed
            # Input lookup and transfer are included in training time.
            index=int(order[updates%len(order)])
            state,metrics=jax.block_until_ready(training_executable(state,batch_at(training,index)))
            metrics={key:np.asarray(value).item() for key,value in metrics.items()}
            if not metrics["finite"]:
                raise FloatingPointError("Nonfinite loss/gradient; the affected weight update was not applied.")
            updates+=1;tokens_processed+=int(metrics["tokens"])
            metrics.update(update=updates,start_seconds=last_start,end_seconds=time.monotonic()-started)
            log.write(json.dumps(metrics,allow_nan=False)+"\n");log.flush()
            if updates<=5 or updates%10==0:
                print(f"update={updates} loss={metrics['loss']:.5f} elapsed={metrics['end_seconds']:.2f}s",flush=True)
    jax.block_until_ready(state)
    training_seconds=time.monotonic()-started
    changed=not np.array_equal(initial_probe,probe(state.params))
    phase=time.monotonic();after=evaluate(state.params);final_eval_seconds=time.monotonic()-phase
    metadata={"config":config,"parameter_count":parameter_count,"data_manifest_sha256":manifest_hash,
              "updates":updates,"model_initialization":"random","cpu_test":cpu_test,
              "model_implementation":"gemma==4.0.1; gm.nn.Gemma3_270M"}
    checkpoint_started=time.monotonic()
    local_checkpoint=work/"checkpoint.npz"
    print("Saving full resumable checkpoint (approximately 3.2 GB for the native model)...",flush=True)
    leaf_count=save_archive(local_checkpoint,state,metadata)
    checkpoint=output/"checkpoint.npz"
    shutil.copyfile(local_checkpoint,checkpoint)
    print("Verifying persistent checkpoint against every state array...",flush=True)
    verified_leaves=verify_archive(checkpoint,state)
    checkpoint_sha256=sha256_file(checkpoint)
    # Shape-only template retains no old device buffers during restoration.
    template=jax.tree.map(lambda leaf:jax.ShapeDtypeStruct(leaf.shape,leaf.dtype),state)
    saved_step=int(state.step);saved_position=int(state.data_position)
    del state
    gc.collect()
    state=restore_archive(checkpoint,template)
    jax.block_until_ready(state)
    if int(state.step)!=saved_step or int(state.data_position)!=saved_position:
        raise ValueError("Restored step/data cursor differs.")
    reloaded=evaluate(state.params)
    reload_error=abs(after["loss"]-reloaded["loss"])
    checkpoint_seconds=time.monotonic()-checkpoint_started
    memory_stats=jax.devices()[0].memory_stats()
    shutil.copyfile(log_path,output/"training.jsonl")
    shutil.copyfile(data_dir/"manifest.json",output/"data_manifest.json")
    if (data_dir/"train.jsonl").exists():
        import gzip
        with gzip.open(output/"source_documents.jsonl.gz","wb") as archive:
            for filename in ("validation.jsonl","train.jsonl"):
                with (data_dir/filename).open("rb") as source:
                    shutil.copyfileobj(source,archive)
    checks={"COLAB-001_SINGLE_V5E":environment["platform"]=="tpu" and environment["device_count"]==1,
        "COLAB-002_NATIVE_RANDOM":not cpu_test and parameter_count==268_098_176,
        "COLAB-003_TIED_EMBEDDING":tied,
        "COLAB-004_DISJOINT_DOCUMENTS":not bool(train_hashes&eval_hashes),
        "COLAB-005_TIMED_TRAINING":training_seconds>=config["training_seconds"] and last_start<config["training_seconds"],
        "COLAB-006_MINIMUM_UPDATES":updates>=config["minimum_updates"],
        "COLAB-007_FINITE_UPDATES":True,
        "COLAB-008_TRANSFORMER_CHANGED":changed,
        "COLAB-009_SAME_VALIDATION":before["manifest_sha256"]==after["manifest_sha256"],
        "COLAB-010_EXACT_CHECKPOINT":verified_leaves==leaf_count,
        "COLAB-011_RELOAD_LOSS":reload_error<=config["reload_loss_tolerance"],
        "COLAB-012_ZERO_UNTIMED_UPDATES":True}
    result={"config":config,"cpu_test":cpu_test,"device":environment["device_kind"],"model_parameters":parameter_count,
        "initialization_seconds":initialization_seconds,"compilation_seconds":compilation_seconds,
        "initial_eval_seconds":initial_eval_seconds,"final_eval_seconds":final_eval_seconds,
        "training_seconds":training_seconds,"training_overrun_seconds":training_seconds-config["training_seconds"],
        "last_update_start_seconds":last_start,"updates":updates,"scored_tokens":tokens_processed,
        "tokens_per_second":tokens_processed/training_seconds,"data_passes":updates//len(order),
        "initial_eval":before,"final_eval":after,"reloaded_eval":reloaded,"reload_loss_error":reload_error,
        "checkpoint_save_verify_reload_seconds":checkpoint_seconds,"checkpoint_bytes":checkpoint.stat().st_size,
        "checkpoint_sha256":checkpoint_sha256,"checkpoint_array_leaves":leaf_count,
        "memory_stats":memory_stats,"memory_scope":"Device-lifetime peak, including initialization and evaluation.",
        "notebook_elapsed_seconds":time.monotonic()-config["notebook_start_monotonic"],
        "cost_usd":None,"cost_note":"Colab compute-unit usage is shown by Colab; this notebook does not invent a USD rate.",
        "requirements":{key:"PASS" if passed else "FAIL" for key,passed in checks.items()},
        "engineering_status":"PASS" if all(checks.values()) else "FAIL",
        "learning_status":"PASS" if after["loss"]<before["loss"] else "FAIL"}
    write_json(output/"result.json",result)
    print(json.dumps({key:result[key] for key in ("engineering_status","learning_status","updates","training_seconds","tokens_per_second","reload_loss_error")},indent=2),flush=True)
    if cpu_test:
        return all(value for key,value in checks.items() if key not in {"COLAB-001_SINGLE_V5E","COLAB-002_NATIVE_RANDOM"})
    return all(checks.values())


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument("--phase",choices=["preflight","prepare","train","cpu-test"],required=True)
    parser.add_argument("--work",type=Path,required=True)
    parser.add_argument("--output",type=Path,required=True)
    args=parser.parse_args()
    args.work.mkdir(parents=True,exist_ok=True);args.output.mkdir(parents=True,exist_ok=True)
    try:
        if args.phase=="preflight":
            info=preflight();write_json(args.output/"preflight.json",info)
            print(json.dumps({key:info[key] for key in ("platform","device_kind","device_count","jax")},indent=2))
        elif args.phase=="prepare":
            config=read_json(args.work/"config.json")
            prepare(config,args.work/"prepared")
        elif args.phase=="cpu-test":
            config=read_json(args.work/"config.json")
            create_fixture(args.work/"prepared",config)
            if not train_pilot(args.work,args.output,cpu_test=True):
                raise RuntimeError("CPU fixture failed a non-hardware requirement.")
        elif not train_pilot(args.work,args.output):
            raise RuntimeError("Engineering criteria failed; inspect result.json.")
    except Exception:
        write_json(args.output/"failure.json",{"phase":args.phase,"error":traceback.format_exc()})
        raise

if __name__=="__main__":
    main()

'''

PROGRAM_PATH = WORK / "pilot.py"
PROGRAM_PATH.write_text(PROGRAM_SOURCE)

CONFIG = {'experiment': 'GEM-COLAB-001', 'model_id': 'google/gemma-3-270m', 'model_revision': 'main', 'dataset_id': 'HuggingFaceFW/fineweb', 'dataset_config': 'sample-10BT', 'dataset_revision': 'main', 'train_documents': 10000, 'validation_documents': 32, 'sequence_length': 1024, 'seed': 42, 'expected_devices': 1, 'local_batch_per_chip': 1, 'training_seconds': 60.0, 'minimum_updates': 5, 'learning_rate': 0.0003, 'warmup_updates': 5, 'adam_b1': 0.9, 'adam_b2': 0.999, 'adam_eps': 1e-08, 'weight_decay': 0.01, 'gradient_clip': 1.0, 'reload_loss_tolerance': 0.001}

CONFIG["notebook_start_monotonic"] = NOTEBOOK_START_MONOTONIC
CONFIG["setup_seconds"] = SETUP_SECONDS
CONFIG_PATH = WORK / "config.json"
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2))
shutil.copyfile(CONFIG_PATH, OUTPUT / "requested_config.json")
shutil.copyfile(LOCK_PATH, OUTPUT / "requirements.lock")
shutil.copyfile(PROGRAM_PATH, OUTPUT / "pilot_source.py")

def run_phase(phase, token=None):
    environment = dict(os.environ)
    environment.pop("PYTHONPATH", None)
    environment.pop("HF_TOKEN", None)
    if token:
        environment["HF_TOKEN"] = token
    environment["TOKENIZERS_PARALLELISM"] = "false"
    environment["PYTHONUNBUFFERED"] = "1"
    environment["MPLCONFIGDIR"] = str(BASE / "matplotlib-cache")
    command = [str(PYTHON), str(PROGRAM_PATH), "--phase", phase,
               "--work", str(WORK), "--output", str(OUTPUT)]
    with (OUTPUT / f"{phase}.log").open("w") as log:
        process = subprocess.Popen(command, env=environment, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        status = process.wait()
    if status:
        raise RuntimeError(f"Phase {phase} failed. See {OUTPUT / (phase + '.log')} and failure.json.")


## 4. Verify the attached accelerator
The program must report **one v5e TPU device** before downloading training data.

In [ ]:
run_phase("preflight")


## 5. Freeze and tokenize the data
Source revisions are resolved to immutable commits. Documents are split before chunking; long documents are processed in successive chunks, with padding excluded from loss.

In [ ]:
PREPARATION_START = time.monotonic()
run_phase("prepare", token=HF_TOKEN)
HF_TOKEN = None
PREPARATION_SECONDS = time.monotonic() - PREPARATION_START
manifest = json.loads((WORK / "prepared" / "manifest.json").read_text())
print(json.dumps({
    "model_revision": manifest["model_revision"],
    "dataset_revision": manifest["dataset_revision"],
    "training_documents": len(manifest["splits"]["train"]["documents"]),
    "validation_documents": len(manifest["splits"]["validation"]["documents"]),
    "preparation_seconds": PREPARATION_SECONDS,
}, indent=2))


## 6. Run the 60-second training test
This cell also initializes, compiles, evaluates, saves, reloads, and verifies. Keep the runtime connected until it finishes saving to Drive. Only optimizer updates belong to the 60-second interval.

In [ ]:
run_phase("train")


## 7. Read the measured results
Learning improvement is reported separately from engineering correctness. No TPU performance numbers are prefilled.

In [ ]:
from IPython.display import display, Markdown
result = json.loads((OUTPUT / "result.json").read_text())
rows = [
    ("Engineering", result["engineering_status"]),
    ("Learning", result["learning_status"]),
    ("Device", result["device"]),
    ("Parameters", f'{result["model_parameters"]:,}'),
    ("Setup seconds", f'{SETUP_SECONDS:.1f}'),
    ("Data preparation seconds", f'{PREPARATION_SECONDS:.1f}'),
    ("Initialization seconds", f'{result["initialization_seconds"]:.2f}'),
    ("Compilation seconds", f'{result["compilation_seconds"]:.2f}'),
    ("Actual training seconds", f'{result["training_seconds"]:.3f}'),
    ("Completed updates", result["updates"]),
    ("Scored tokens/second", f'{result["tokens_per_second"]:,.1f}'),
    ("Initial validation loss", f'{result["initial_eval"]["loss"]:.6f}'),
    ("Final validation loss", f'{result["final_eval"]["loss"]:.6f}'),
    ("Reload loss error", f'{result["reload_loss_error"]:.8f}'),
    ("Full checkpoint size GB", f'{result["checkpoint_bytes"]/1e9:.3f}'),
    ("Save/verify/reload seconds", f'{result["checkpoint_save_verify_reload_seconds"]:.1f}'),
]
display(Markdown("| Measurement | Result |\n|---|---|\n" + "\n".join(f"| {name} | {value} |" for name,value in rows)))
print(json.dumps(result["requirements"], indent=2))
print("All artifacts:", OUTPUT)
print("Colab compute-unit usage is displayed by Colab; no USD rate is assumed.")


## 8. Download the small report bundle
The approximately 3.2 GB checkpoint stays in your Google Drive run folder. This download includes the report, logs, manifest, dependency lock, and embedded source.

In [ ]:
import zipfile
from google.colab import files
REPORT_ZIP = WORK / "gemma_colab_pilot_reports.zip"
with zipfile.ZipFile(REPORT_ZIP, "w", zipfile.ZIP_DEFLATED) as bundle:
    for filename in ["result.json", "environment.json", "preflight.json", "requested_config.json",
                     "config.json", "data_manifest.json", "training.jsonl", "requirements.lock",
                     "pilot_source.py", "preflight.log", "prepare.log", "train.log"]:
        path = OUTPUT / filename
        if path.is_file():
            bundle.write(path, filename)
shutil.copyfile(REPORT_ZIP, OUTPUT / REPORT_ZIP.name)
files.download(str(REPORT_ZIP))
print("Saved checkpoint:", OUTPUT / "checkpoint.npz")
print("Finished. Use Runtime > Disconnect and delete runtime when you are done.")


## Reproducibility notes

- The persistent checkpoint includes parameters, AdamW moments and counts, PRNG state, data position, configuration, and the data-manifest hash.
- Saving and comparison process **one array at a time**. The previous device state is released before reconstruction to reduce checkpoint memory pressure.
- Model/data artifacts are pinned during preparation; original selected documents are also saved as a compressed text file in Drive.
- Tokenizer round-trip counts are recorded. This pilot reports native-tokenizer cross-entropy, including EOS targets; it does not silently present this as byte-identical tokenizer-comparison BPB.
- Peak device memory, when available from JAX, covers the device lifetime rather than only the training interval.
- This notebook has a local CPU fixture for development. That fixture uses a reduced model and synthetic data and is never reported as a real TPU PASS.

Sources: [native Gemma API](https://gemma-llm.readthedocs.io/en/latest/api/gm/nn/Gemma3_270M.html), [JAX installation](https://docs.jax.dev/en/latest/installation.html), [FineWeb](https://huggingface.co/datasets/HuggingFaceFW/fineweb), [NASA requirements guidance](https://www.nasa.gov/reference/appendix-c-how-to-write-a-good-requirement/).
